# 01 — Descriptive and Inequality Analysis (Revisi v2)

Notebook ini menerjemahkan dataset yang telah lolos audit menjadi bukti deskriptif yang siap digunakan untuk artikel.

## Tujuan

1. Mendeskripsikan distribusi jumlah lulusan, mahasiswa terdaftar/baru, kapasitas perguruan tinggi, dan mutu dosen.
2. Mengukur konsentrasi dan ketimpangan antarprovinsi serta antarkabupaten/kota menggunakan Gini, Theil T, HHI, dan *top-share*.
3. Memisahkan ketimpangan jumlah absolut dari ketimpangan intensitas keluaran.
4. Mengidentifikasi kesenjangan gender, konsentrasi STEM, dan pencilan wilayah.
5. Menilai implikasi distribusi data untuk pemodelan Negative Binomial dan validasi berbasis provinsi.

> **Batas interpretasi:** seluruh analisis bersifat agregat, ekologis, dan *cross-sectional*. Rasio lulusan terhadap mahasiswa terdaftar/baru bukan *completion rate* cohort.

> **Revisi v2:** fungsi statistik deskriptif ditanamkan langsung dalam notebook. Notebook tidak lagi bergantung pada modul lokal `descriptive_metrics.py`, dan lokasi proyek/dataset dicari secara lebih robust.


In [ ]:
from __future__ import annotations

import json
import platform
import random
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

print("Python     :", sys.version.split()[0])
print("Pandas     :", pd.__version__)
print("NumPy      :", np.__version__)
print("Matplotlib :", plt.matplotlib.__version__)
print("Platform   :", platform.platform())

In [ ]:
# Resolusi lokasi proyek yang robust.
# Notebook dapat dijalankan dari folder proyek, folder notebooks, atau Jupyter yang
# diluncurkan dari satu tingkat di atas folder proyek.
CURRENT_DIR = Path.cwd().resolve()
WORKBOOK_NAME = "Dataset_Gabungan_Kemdiktisaintek_2025.xlsx"


def _candidate_project_roots(start: Path) -> list[Path]:
    candidates: list[Path] = []

    def add(path: Path) -> None:
        path = path.resolve()
        if path not in candidates:
            candidates.append(path)

    add(start)
    for parent in start.parents:
        add(parent)

    # Folder proyek yang berada langsung di bawah working directory.
    try:
        for child in start.iterdir():
            if child.is_dir():
                add(child)
    except PermissionError:
        pass

    # Nama folder proyek yang lazim digunakan pada paket ini.
    for base in list(candidates):
        add(base / "pt_education_research_project")
        add(base / "pt_education_research_project_00_01")
        add(base / "pt_education_research_project_00_01_v2")

    return candidates


PROJECT_ROOT = None
COMBINED_FILE = None

for candidate in _candidate_project_roots(CURRENT_DIR):
    workbook_candidate = candidate / "data" / "processed" / WORKBOOK_NAME
    if workbook_candidate.exists():
        PROJECT_ROOT = candidate
        COMBINED_FILE = workbook_candidate
        break

# Fallback: workbook ditempatkan pada working directory atau satu tingkat di atasnya.
if COMBINED_FILE is None:
    for candidate in [CURRENT_DIR, CURRENT_DIR.parent]:
        workbook_candidate = candidate / WORKBOOK_NAME
        if workbook_candidate.exists():
            PROJECT_ROOT = candidate
            COMBINED_FILE = workbook_candidate
            break

if PROJECT_ROOT is None or COMBINED_FILE is None:
    searched = "\n".join(
        str(path / "data" / "processed" / WORKBOOK_NAME)
        for path in _candidate_project_roots(CURRENT_DIR)[:12]
    )
    raise FileNotFoundError(
        "Dataset gabungan tidak ditemukan. Ekstrak paket proyek lengkap, lalu jalankan "
        "Jupyter dari folder pt_education_research_project. Lokasi yang diperiksa antara lain:\n"
        + searched
    )

PROCESSED_DIR = COMBINED_FILE.parent
OUTPUT_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

VALIDATED_GENDER_FILE = OUTPUT_TABLE_DIR / "00_validated_AD_Gender_Jenjang.csv"
VALIDATED_STEM_FILE = OUTPUT_TABLE_DIR / "00_validated_AD_STEM_Jenjang.csv"
READINESS_FILE = OUTPUT_TABLE_DIR / "00_readiness_checklist.csv"

OUTPUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("CURRENT_DIR :", CURRENT_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("COMBINED_FILE:", COMBINED_FILE)


In [ ]:
# Fungsi statistik ditanamkan langsung agar notebook dapat dijalankan mandiri.
from typing import Iterable, Sequence
from scipy.stats import spearmanr


def safe_divide(numerator, denominator):
    num = pd.to_numeric(numerator, errors="coerce")
    den = pd.to_numeric(denominator, errors="coerce")
    return num.div(den.where(den.ne(0)))


def gini_coefficient(values: Iterable[float]) -> float:
    x = np.asarray(list(values), dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    if np.any(x < 0):
        raise ValueError("Gini hanya menerima nilai nonnegatif.")
    total = x.sum()
    if total == 0:
        return 0.0
    x = np.sort(x)
    n = x.size
    index = np.arange(1, n + 1)
    return float((2 * np.sum(index * x) / (n * total)) - (n + 1) / n)


def theil_t(values: Iterable[float]) -> float:
    x = np.asarray(list(values), dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    if np.any(x < 0):
        raise ValueError("Theil T hanya menerima nilai nonnegatif.")
    total = x.sum()
    if total == 0:
        return 0.0
    positive = x[x > 0]
    shares = positive / total
    return float(np.sum(shares * np.log(shares * x.size)))


def herfindahl_hirschman(values: Iterable[float]) -> float:
    x = np.asarray(list(values), dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    total = x.sum()
    if total == 0:
        return 0.0
    shares = x / total
    return float(np.sum(shares**2))


def concentration_share(values: Iterable[float], top_n: int) -> float:
    x = np.asarray(list(values), dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    total = x.sum()
    if total == 0:
        return 0.0
    top_n = min(int(top_n), x.size)
    return float(np.sort(x)[-top_n:].sum() / total)


def lorenz_coordinates(values: Iterable[float]) -> tuple[np.ndarray, np.ndarray]:
    x = np.asarray(list(values), dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.array([0.0, 1.0]), np.array([0.0, 1.0])
    if np.any(x < 0):
        raise ValueError("Kurva Lorenz hanya menerima nilai nonnegatif.")
    x = np.sort(x)
    total = x.sum()
    population = np.arange(0, x.size + 1) / x.size
    outcome = population.copy() if total == 0 else np.insert(np.cumsum(x) / total, 0, 0.0)
    return population, outcome


def inequality_summary(values: Iterable[float], label: str) -> dict:
    x = np.asarray(list(values), dtype=float)
    x = x[np.isfinite(x)]
    return {
        "indicator": label,
        "n_units": int(x.size),
        "total": float(x.sum()) if x.size else np.nan,
        "mean": float(x.mean()) if x.size else np.nan,
        "median": float(np.median(x)) if x.size else np.nan,
        "gini": gini_coefficient(x),
        "theil_t": theil_t(x),
        "hhi": herfindahl_hirschman(x),
        "top_1_share": concentration_share(x, 1),
        "top_5_share": concentration_share(x, 5),
        "top_10_share": concentration_share(x, 10),
    }


def bootstrap_inequality(
    values: Iterable[float], *, n_boot: int = 1000, random_state: int = 42
) -> dict:
    x = np.asarray(list(values), dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return {
            "gini_lower": np.nan,
            "gini_upper": np.nan,
            "theil_lower": np.nan,
            "theil_upper": np.nan,
        }
    rng = np.random.default_rng(random_state)
    gini_boot = np.empty(n_boot)
    theil_boot = np.empty(n_boot)
    for i in range(n_boot):
        sample = rng.choice(x, size=x.size, replace=True)
        gini_boot[i] = gini_coefficient(sample)
        theil_boot[i] = theil_t(sample)
    return {
        "gini_lower": float(np.quantile(gini_boot, 0.025)),
        "gini_upper": float(np.quantile(gini_boot, 0.975)),
        "theil_lower": float(np.quantile(theil_boot, 0.025)),
        "theil_upper": float(np.quantile(theil_boot, 0.975)),
    }


def theil_decomposition(frame: pd.DataFrame, *, value_col: str, group_col: str) -> dict:
    data = frame[[group_col, value_col]].copy()
    data[value_col] = pd.to_numeric(data[value_col], errors="coerce")
    data = data.dropna(subset=[group_col, value_col])
    data = data.loc[data[value_col] >= 0]
    n_total = len(data)
    y_total = data[value_col].sum()
    total_theil = theil_t(data[value_col])
    if n_total == 0 or y_total == 0:
        return {
            "theil_total": total_theil,
            "theil_between": 0.0,
            "theil_within": 0.0,
            "between_share": 0.0,
            "within_share": 0.0,
            "decomposition_error": 0.0,
        }
    between = 0.0
    within = 0.0
    for _, group in data.groupby(group_col, dropna=False):
        n_g = len(group)
        y_g = group[value_col].sum()
        if y_g <= 0:
            continue
        income_share = y_g / y_total
        population_share = n_g / n_total
        between += income_share * np.log(income_share / population_share)
        within += income_share * theil_t(group[value_col])
    return {
        "theil_total": float(total_theil),
        "theil_between": float(between),
        "theil_within": float(within),
        "between_share": float(between / total_theil) if total_theil > 0 else 0.0,
        "within_share": float(within / total_theil) if total_theil > 0 else 0.0,
        "decomposition_error": float(total_theil - between - within),
    }


def robust_zscore(values: Iterable[float]) -> np.ndarray:
    x = np.asarray(list(values), dtype=float)
    result = np.full(x.shape, np.nan, dtype=float)
    finite = np.isfinite(x)
    if not finite.any():
        return result
    median = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - median))
    if mad == 0:
        return result
    result[finite] = 0.67448975 * (x[finite] - median) / mad
    return result


def benjamini_hochberg(p_values: Sequence[float]) -> np.ndarray:
    p = np.asarray(p_values, dtype=float)
    q = np.full_like(p, np.nan, dtype=float)
    finite_idx = np.flatnonzero(np.isfinite(p))
    if finite_idx.size == 0:
        return q
    p_finite = p[finite_idx]
    order = np.argsort(p_finite)
    ranked = p_finite[order]
    m = len(ranked)
    adjusted = ranked * m / np.arange(1, m + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    q_finite = np.empty_like(adjusted)
    q_finite[order] = adjusted
    q[finite_idx] = q_finite
    return q


def spearman_target_table(
    frame: pd.DataFrame, *, targets: Sequence[str], predictors: Sequence[str]
) -> pd.DataFrame:
    rows = []
    for target in targets:
        for predictor in predictors:
            pair = frame[[target, predictor]].replace([np.inf, -np.inf], np.nan).dropna()
            if len(pair) < 3 or pair[target].nunique() < 2 or pair[predictor].nunique() < 2:
                rho, p_value = np.nan, np.nan
            else:
                result = spearmanr(pair[target], pair[predictor])
                rho, p_value = float(result.statistic), float(result.pvalue)
            rows.append(
                {
                    "target": target,
                    "predictor": predictor,
                    "n_complete": len(pair),
                    "spearman_rho": rho,
                    "p_value": p_value,
                }
            )
    result = pd.DataFrame(rows)
    result["q_value_bh"] = np.nan
    for _, indices in result.groupby("target").groups.items():
        result.loc[indices, "q_value_bh"] = benjamini_hochberg(
            result.loc[indices, "p_value"].to_numpy()
        )
    return result


def descriptive_table(frame: pd.DataFrame, columns: Sequence[str]) -> pd.DataFrame:
    rows = []
    for col in columns:
        x = pd.to_numeric(frame[col], errors="coerce")
        valid = x.dropna()
        mean = valid.mean() if len(valid) else np.nan
        variance = valid.var(ddof=1) if len(valid) > 1 else np.nan
        rows.append(
            {
                "variable": col,
                "n": int(valid.size),
                "missing_n": int(x.isna().sum()),
                "missing_pct": float(x.isna().mean() * 100),
                "mean": float(mean) if pd.notna(mean) else np.nan,
                "std": float(valid.std(ddof=1)) if len(valid) > 1 else np.nan,
                "median": float(valid.median()) if len(valid) else np.nan,
                "q1": float(valid.quantile(0.25)) if len(valid) else np.nan,
                "q3": float(valid.quantile(0.75)) if len(valid) else np.nan,
                "p95": float(valid.quantile(0.95)) if len(valid) else np.nan,
                "p99": float(valid.quantile(0.99)) if len(valid) else np.nan,
                "min": float(valid.min()) if len(valid) else np.nan,
                "max": float(valid.max()) if len(valid) else np.nan,
                "skewness": float(valid.skew()) if len(valid) > 2 else np.nan,
                "variance_to_mean": (
                    float(variance / mean)
                    if pd.notna(variance) and pd.notna(mean) and mean != 0
                    else np.nan
                ),
            }
        )
    return pd.DataFrame(rows)


print("PASS — fungsi statistik deskriptif tersedia secara mandiri.")


## 1. Memuat dataset tervalidasi

Notebook memprioritaskan CSV hasil Notebook 00. Apabila CSV tersebut belum berada di folder proyek, notebook membaca workbook gabungan dan menghitung kembali seluruh indikator rasio.

In [ ]:
def standardize_columns(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result.columns = (
        pd.Index(result.columns)
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[^0-9a-zA-Z]+", "_", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )
    return result


def recompute_gender_indicators(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result["jumlah_aktif_total"] = result[
        ["jumlah_aktif_a", "jumlah_mbkm_m"]
    ].sum(axis=1, min_count=1)
    result["graduate_output_intensity"] = safe_divide(
        result["jumlah_lulusan"], result["jumlah_terdaftar"]
    )
    result["lulusan_per_1000_terdaftar"] = (
        result["graduate_output_intensity"] * 1000
    )
    result["proporsi_mbkm"] = safe_divide(
        result["jumlah_mbkm_m"], result["jumlah_aktif_total"]
    )
    result["proporsi_pt_akreditasi_tinggi"] = safe_divide(
        result["jumlah_pt_akreditasi_tinggi"], result["jumlah_pt"]
    )
    result["proporsi_profesor"] = safe_divide(
        result["dosen_profesor"], result["dosen_total_jabatan"]
    )
    result["proporsi_dosen_senior"] = safe_divide(
        result["dosen_senior"], result["dosen_total_jabatan"]
    )
    result["proporsi_dosen_tersertifikasi"] = safe_divide(
        result["dosen_tersertifikasi"], result["dosen_total_sertifikasi"]
    )
    if {"dosen_jk_profesor", "dosen_jk_total_jabatan"}.issubset(result.columns):
        result["proporsi_profesor_jk"] = safe_divide(
            result["dosen_jk_profesor"], result["dosen_jk_total_jabatan"]
        )
    if {"dosen_jk_tersertifikasi", "dosen_jk_total_sertifikasi"}.issubset(result.columns):
        result["proporsi_sertifikasi_jk"] = safe_divide(
            result["dosen_jk_tersertifikasi"],
            result["dosen_jk_total_sertifikasi"],
        )
    return result


def recompute_stem_indicators(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result["graduate_to_entry_flow_ratio"] = safe_divide(
        result["jumlah_lulusan"], result["jumlah_mahasiswa_baru"]
    )
    result["lulusan_per_100_mahasiswa_baru"] = (
        result["graduate_to_entry_flow_ratio"] * 100
    )
    result["proporsi_mhs_baru_pt_akreditasi_tinggi"] = safe_divide(
        result["mhs_baru_pt_akreditasi_tinggi"],
        result["jumlah_mahasiswa_baru"],
    )
    result["proporsi_mhs_baru_prodi_akreditasi_tinggi"] = safe_divide(
        result["mhs_baru_prodi_akreditasi_tinggi"],
        result["jumlah_mahasiswa_baru"],
    )
    result["proporsi_mhs_baru_pt_prodi_tinggi"] = safe_divide(
        result["mhs_baru_pt_prodi_tinggi"],
        result["jumlah_mahasiswa_baru"],
    )
    result["proporsi_pt_akreditasi_tinggi"] = safe_divide(
        result["jumlah_pt_akreditasi_tinggi"], result["jumlah_pt"]
    )
    result["proporsi_profesor"] = safe_divide(
        result["dosen_profesor"], result["dosen_total_jabatan"]
    )
    result["proporsi_dosen_senior"] = safe_divide(
        result["dosen_senior"], result["dosen_total_jabatan"]
    )
    result["proporsi_dosen_tersertifikasi"] = safe_divide(
        result["dosen_tersertifikasi"], result["dosen_total_sertifikasi"]
    )
    result["proporsi_doktor_stem"] = safe_divide(
        result["dosen_doktor_stem"], result["dosen_doktor_total"]
    )
    return result


if VALIDATED_GENDER_FILE.exists() and VALIDATED_STEM_FILE.exists():
    DATA_SOURCE = "CSV tervalidasi hasil Notebook 00"
    gender_df = pd.read_csv(VALIDATED_GENDER_FILE)
    stem_df = pd.read_csv(VALIDATED_STEM_FILE)
else:
    DATA_SOURCE = "Workbook gabungan — fallback"
    assert COMBINED_FILE.exists(), f"File tidak ditemukan: {COMBINED_FILE}"
    gender_df = pd.read_excel(
        COMBINED_FILE, sheet_name="AD_Gender_Jenjang", engine="openpyxl"
    )
    stem_df = pd.read_excel(
        COMBINED_FILE, sheet_name="AD_STEM_Jenjang", engine="openpyxl"
    )

gender_df = recompute_gender_indicators(standardize_columns(gender_df))
stem_df = recompute_stem_indicators(standardize_columns(stem_df))

print("Sumber data:", DATA_SOURCE)
print("Dataset A  :", gender_df.shape)
print("Dataset B  :", stem_df.shape)

In [ ]:
KEY_A = ["provinsi_pt", "kabupaten_pt", "jk", "nm_jenj_didik"]
KEY_B = ["provinsi_pt", "kabupaten_pt", "a_stem", "nm_jenj_didik"]

required_a = {
    *KEY_A,
    "jumlah_lulusan",
    "jumlah_terdaftar",
    "jumlah_aktif_a",
    "jumlah_mbkm_m",
    "jumlah_pt",
    "jumlah_pt_akreditasi_tinggi",
    "dosen_total_jabatan",
    "dosen_profesor",
    "dosen_senior",
    "dosen_total_sertifikasi",
    "dosen_tersertifikasi",
}
required_b = {
    *KEY_B,
    "jumlah_lulusan",
    "jumlah_mahasiswa_baru",
    "dosen_doktor_stem",
    "dosen_doktor_non_stem",
    "dosen_doktor_total",
}

assert required_a.issubset(gender_df.columns), sorted(required_a - set(gender_df.columns))
assert required_b.issubset(stem_df.columns), sorted(required_b - set(stem_df.columns))
assert not gender_df.duplicated(KEY_A).any(), "Dataset A memiliki kunci duplikat."
assert not stem_df.duplicated(KEY_B).any(), "Dataset B memiliki kunci duplikat."
assert gender_df["jumlah_lulusan"].ge(0).all()
assert stem_df["jumlah_lulusan"].ge(0).all()

if READINESS_FILE.exists():
    readiness_00 = pd.read_csv(READINESS_FILE)
    assert readiness_00["status"].eq("PASS").all(), "Notebook 00 belum seluruhnya PASS."

print("PASS — validasi minimum Notebook 01 terpenuhi.")

## 2. Definisi populasi analisis dan cakupan data

Analisis jumlah lulusan menggunakan seluruh baris outcome. Analisis rasio hanya menggunakan baris dengan denominator positif. Analisis prediktor institusional menggunakan baris dengan flag sumber yang sesuai.

In [ ]:
coverage_rows = []
for dataset_name, frame, denominator, flag_columns in [
    (
        "AD_Gender_Jenjang",
        gender_df,
        "jumlah_terdaftar",
        ["flag_match_terdaftar", "flag_match_aktif", "flag_match_pt", "flag_match_dosen"],
    ),
    (
        "AD_STEM_Jenjang",
        stem_df,
        "jumlah_mahasiswa_baru",
        [
            "flag_match_mahasiswa_baru",
            "flag_match_pt",
            "flag_match_dosen_doktor",
            "flag_match_dosen",
        ],
    ),
]:
    row = {
        "dataset": dataset_name,
        "rows": len(frame),
        "provinces": frame["provinsi_pt"].nunique(),
        "districts": frame[["provinsi_pt", "kabupaten_pt"]].drop_duplicates().shape[0],
        "total_graduates": frame["jumlah_lulusan"].sum(),
        "denominator_available_n": frame[denominator].notna().sum(),
        "denominator_positive_n": frame[denominator].gt(0).sum(),
    }
    for flag in flag_columns:
        if flag in frame.columns:
            row[f"{flag}_match_pct"] = frame[flag].eq("MATCH").mean() * 100
    coverage_rows.append(row)

coverage_summary = pd.DataFrame(coverage_rows)
coverage_summary.to_csv(OUTPUT_TABLE_DIR / "01_data_coverage.csv", index=False)
display(coverage_summary)

## 3. Statistik deskriptif tingkat strata

Ukuran dispersi terutama digunakan untuk memeriksa *right skewness* dan *overdispersion*. Rasio `variance_to_mean` yang jauh di atas satu menjadi indikasi awal bahwa distribusi Poisson sederhana tidak memadai.

In [ ]:
numeric_a = [
    "jumlah_lulusan",
    "jumlah_terdaftar",
    "jumlah_aktif_a",
    "jumlah_mbkm_m",
    "graduate_output_intensity",
    "proporsi_mbkm",
    "jumlah_pt",
    "proporsi_pt_akreditasi_tinggi",
    "proporsi_profesor",
    "proporsi_dosen_senior",
    "proporsi_dosen_tersertifikasi",
]
numeric_b = [
    "jumlah_lulusan",
    "jumlah_mahasiswa_baru",
    "graduate_to_entry_flow_ratio",
    "proporsi_mhs_baru_pt_akreditasi_tinggi",
    "proporsi_mhs_baru_prodi_akreditasi_tinggi",
    "jumlah_pt",
    "proporsi_pt_akreditasi_tinggi",
    "dosen_doktor_sesuai_bidang",
    "proporsi_doktor_stem",
]

descriptive_a = descriptive_table(gender_df, numeric_a).assign(dataset="AD_Gender_Jenjang")
descriptive_b = descriptive_table(stem_df, numeric_b).assign(dataset="AD_STEM_Jenjang")
descriptive_strata = pd.concat([descriptive_a, descriptive_b], ignore_index=True)

descriptive_strata.to_csv(
    OUTPUT_TABLE_DIR / "01_descriptive_statistics_strata.csv", index=False
)
display(descriptive_strata)

In [ ]:
def save_figure(filename: str) -> None:
    plt.tight_layout()
    plt.savefig(OUTPUT_FIGURE_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

plt.figure(figsize=(8, 5))
plt.hist(gender_df["jumlah_lulusan"].dropna(), bins=50)
plt.xlabel("Jumlah lulusan per strata")
plt.ylabel("Frekuensi")
plt.title("Distribusi jumlah lulusan pada skala asli")
save_figure("01_distribution_graduates_raw.png")

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(np.log1p(gender_df["jumlah_lulusan"].dropna()), bins=50)
plt.xlabel("log(1 + jumlah lulusan)")
plt.ylabel("Frekuensi")
plt.title("Distribusi jumlah lulusan setelah transformasi log1p")
save_figure("02_distribution_graduates_log1p.png")

## 4. Komposisi nasional menurut gender dan jenjang

Intensitas pada tabel berikut dihitung sebagai rasio total lulusan terhadap total mahasiswa terdaftar, bukan rata-rata rasio per baris.

In [ ]:
def aggregate_summary(frame: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    summary = (
        frame.groupby(group_columns, dropna=False, as_index=False)
        .agg(
            jumlah_lulusan=("jumlah_lulusan", "sum"),
            jumlah_terdaftar=("jumlah_terdaftar", "sum"),
            jumlah_aktif_a=("jumlah_aktif_a", "sum"),
            jumlah_mbkm_m=("jumlah_mbkm_m", "sum"),
        )
    )
    summary["share_lulusan"] = summary["jumlah_lulusan"] / summary["jumlah_lulusan"].sum()
    summary["graduate_output_intensity"] = safe_divide(
        summary["jumlah_lulusan"], summary["jumlah_terdaftar"]
    )
    summary["lulusan_per_1000_terdaftar"] = summary["graduate_output_intensity"] * 1000
    summary["proporsi_mbkm"] = safe_divide(
        summary["jumlah_mbkm_m"],
        summary["jumlah_aktif_a"] + summary["jumlah_mbkm_m"],
    )
    return summary


gender_summary = aggregate_summary(gender_df, ["jk"]).sort_values("jumlah_lulusan", ascending=False)
level_summary = aggregate_summary(gender_df, ["nm_jenj_didik"]).sort_values(
    "jumlah_lulusan", ascending=False
)
gender_level_summary = aggregate_summary(gender_df, ["nm_jenj_didik", "jk"])

gender_summary.to_csv(OUTPUT_TABLE_DIR / "01_summary_by_gender.csv", index=False)
level_summary.to_csv(OUTPUT_TABLE_DIR / "01_summary_by_level.csv", index=False)
gender_level_summary.to_csv(
    OUTPUT_TABLE_DIR / "01_summary_by_gender_and_level.csv", index=False
)

display(gender_summary)
display(level_summary)

In [ ]:
plot_data = level_summary.sort_values("jumlah_lulusan")
plt.figure(figsize=(9, 6))
plt.barh(plot_data["nm_jenj_didik"], plot_data["jumlah_lulusan"])
plt.xlabel("Jumlah lulusan")
plt.ylabel("Jenjang")
plt.title("Jumlah lulusan nasional menurut jenjang")
plt.ticklabel_format(axis="x", style="plain")
save_figure("03_graduates_by_level.png")

In [ ]:
level_gender_pivot = gender_level_summary.pivot(
    index="nm_jenj_didik", columns="jk", values="jumlah_lulusan"
).fillna(0)
level_gender_pivot["female_share"] = safe_divide(
    level_gender_pivot.get("P", 0),
    level_gender_pivot.get("P", 0) + level_gender_pivot.get("L", 0),
)
level_gender_pivot = level_gender_pivot.join(
    level_summary.set_index("nm_jenj_didik")["jumlah_lulusan"], how="left"
).sort_values("jumlah_lulusan")

plt.figure(figsize=(9, 6))
plt.barh(level_gender_pivot.index, level_gender_pivot["female_share"] * 100)
plt.axvline(50, linestyle="--")
plt.xlabel("Proporsi perempuan di antara lulusan (%)")
plt.ylabel("Jenjang")
plt.title("Representasi perempuan menurut jenjang")
save_figure("04_female_share_by_level.png")

## 5. Agregasi wilayah yang tidak menggandakan indikator institusional

Variabel perguruan tinggi dan dosen berulang pada setiap kombinasi gender–jenjang. Oleh karena itu, indikator tersebut terlebih dahulu dibuat unik pada `provinsi × kabupaten/kota`, baru kemudian diagregasi ke provinsi.

In [ ]:
GEO_KEYS = ["provinsi_pt", "kabupaten_pt"]
GEO_METRICS = [
    "jumlah_pt",
    "jumlah_pt_akreditasi_tinggi",
    "dosen_total_jabatan",
    "dosen_profesor",
    "dosen_senior",
    "dosen_total_sertifikasi",
    "dosen_tersertifikasi",
]

geo_consistency = gender_df.groupby(GEO_KEYS, dropna=False)[GEO_METRICS].nunique(dropna=False)
assert not geo_consistency.gt(1).any(axis=1).any(), (
    "Indikator wilayah tidak konsisten pada strata yang berulang."
)
geo_unique = gender_df[GEO_KEYS + GEO_METRICS].drop_duplicates(GEO_KEYS)

flow_kab = (
    gender_df.groupby(GEO_KEYS, dropna=False, as_index=False)
    .agg(
        jumlah_lulusan=("jumlah_lulusan", "sum"),
        jumlah_terdaftar=("jumlah_terdaftar", "sum"),
        jumlah_aktif_a=("jumlah_aktif_a", "sum"),
        jumlah_mbkm_m=("jumlah_mbkm_m", "sum"),
    )
)
kabupaten_summary = flow_kab.merge(geo_unique, on=GEO_KEYS, how="left", validate="one_to_one")
kabupaten_summary["graduate_output_intensity"] = safe_divide(
    kabupaten_summary["jumlah_lulusan"], kabupaten_summary["jumlah_terdaftar"]
)
kabupaten_summary["lulusan_per_1000_terdaftar"] = (
    kabupaten_summary["graduate_output_intensity"] * 1000
)
kabupaten_summary["proporsi_mbkm"] = safe_divide(
    kabupaten_summary["jumlah_mbkm_m"],
    kabupaten_summary["jumlah_aktif_a"] + kabupaten_summary["jumlah_mbkm_m"],
)
kabupaten_summary["proporsi_pt_akreditasi_tinggi"] = safe_divide(
    kabupaten_summary["jumlah_pt_akreditasi_tinggi"], kabupaten_summary["jumlah_pt"]
)
kabupaten_summary["proporsi_profesor"] = safe_divide(
    kabupaten_summary["dosen_profesor"], kabupaten_summary["dosen_total_jabatan"]
)
kabupaten_summary["proporsi_dosen_senior"] = safe_divide(
    kabupaten_summary["dosen_senior"], kabupaten_summary["dosen_total_jabatan"]
)
kabupaten_summary["proporsi_dosen_tersertifikasi"] = safe_divide(
    kabupaten_summary["dosen_tersertifikasi"], kabupaten_summary["dosen_total_sertifikasi"]
)

province_flows = (
    kabupaten_summary.groupby("provinsi_pt", dropna=False, as_index=False)
    .agg(
        jumlah_lulusan=("jumlah_lulusan", "sum"),
        jumlah_terdaftar=("jumlah_terdaftar", "sum"),
        jumlah_aktif_a=("jumlah_aktif_a", "sum"),
        jumlah_mbkm_m=("jumlah_mbkm_m", "sum"),
        jumlah_pt=("jumlah_pt", "sum"),
        jumlah_pt_akreditasi_tinggi=("jumlah_pt_akreditasi_tinggi", "sum"),
        dosen_total_jabatan=("dosen_total_jabatan", "sum"),
        dosen_profesor=("dosen_profesor", "sum"),
        dosen_senior=("dosen_senior", "sum"),
        dosen_total_sertifikasi=("dosen_total_sertifikasi", "sum"),
        dosen_tersertifikasi=("dosen_tersertifikasi", "sum"),
        jumlah_kabupaten_kota=("kabupaten_pt", "nunique"),
    )
)
province_summary = province_flows.copy()
province_summary["share_lulusan_nasional"] = (
    province_summary["jumlah_lulusan"] / province_summary["jumlah_lulusan"].sum()
)
province_summary["graduate_output_intensity"] = safe_divide(
    province_summary["jumlah_lulusan"], province_summary["jumlah_terdaftar"]
)
province_summary["lulusan_per_1000_terdaftar"] = (
    province_summary["graduate_output_intensity"] * 1000
)
province_summary["proporsi_mbkm"] = safe_divide(
    province_summary["jumlah_mbkm_m"],
    province_summary["jumlah_aktif_a"] + province_summary["jumlah_mbkm_m"],
)
province_summary["proporsi_pt_akreditasi_tinggi"] = safe_divide(
    province_summary["jumlah_pt_akreditasi_tinggi"], province_summary["jumlah_pt"]
)
province_summary["proporsi_profesor"] = safe_divide(
    province_summary["dosen_profesor"], province_summary["dosen_total_jabatan"]
)
province_summary["proporsi_dosen_senior"] = safe_divide(
    province_summary["dosen_senior"], province_summary["dosen_total_jabatan"]
)
province_summary["proporsi_dosen_tersertifikasi"] = safe_divide(
    province_summary["dosen_tersertifikasi"], province_summary["dosen_total_sertifikasi"]
)

kabupaten_summary.to_csv(OUTPUT_TABLE_DIR / "01_kabupaten_summary.csv", index=False)
province_summary.to_csv(OUTPUT_TABLE_DIR / "01_province_summary.csv", index=False)

display(province_summary.sort_values("jumlah_lulusan", ascending=False).head(10))

In [ ]:
plot_data = province_summary.sort_values("jumlah_lulusan").tail(15)
plt.figure(figsize=(10, 7))
plt.barh(plot_data["provinsi_pt"], plot_data["jumlah_lulusan"])
plt.xlabel("Jumlah lulusan")
plt.ylabel("Provinsi")
plt.title("Lima belas provinsi dengan jumlah lulusan terbesar")
plt.ticklabel_format(axis="x", style="plain")
save_figure("05_top15_provinces_graduates.png")

In [ ]:
plot_data = province_summary.dropna(subset=["lulusan_per_1000_terdaftar"]).sort_values(
    "lulusan_per_1000_terdaftar"
)
plt.figure(figsize=(10, 9))
plt.barh(plot_data["provinsi_pt"], plot_data["lulusan_per_1000_terdaftar"])
plt.xlabel("Lulusan per 1.000 mahasiswa terdaftar")
plt.ylabel("Provinsi")
plt.title("Intensitas keluaran lulusan menurut provinsi")
save_figure("06_province_graduate_output_intensity.png")

## 6. Ketimpangan dan konsentrasi wilayah

- **Gini** mengukur ketidakmerataan distribusi antarunit.
- **Theil T** memungkinkan dekomposisi menjadi komponen antarprovinsi dan dalam-provinsi.
- **HHI** mengukur konsentrasi pangsa.
- *Top-share* menunjukkan proporsi total yang dikuasai unit terbesar.

Interval kepercayaan Gini dan Theil diperoleh melalui *bootstrap* nonparametrik 1.000 replikasi.

In [ ]:
inequality_rows = []
for geography, frame in [
    ("Provinsi", province_summary),
    ("Kabupaten/Kota", kabupaten_summary),
]:
    for variable, label in [
        ("jumlah_lulusan", "Jumlah lulusan"),
        ("jumlah_terdaftar", "Jumlah mahasiswa terdaftar"),
        ("jumlah_pt", "Jumlah perguruan tinggi"),
        (
            "graduate_output_intensity",
            "Intensitas keluaran lulusan — tidak dibobot",
        ),
    ]:
        values = frame[variable].dropna()
        result = inequality_summary(values, f"{geography} — {label}")
        result["geography"] = geography
        result["variable"] = variable
        result["interpretation_note"] = (
            "Distribusi rasio antarunit; bukan distribusi jumlah lulusan"
            if variable == "graduate_output_intensity"
            else "Distribusi jumlah absolut antarunit"
        )
        result.update(
            bootstrap_inequality(
                values, n_boot=1000, random_state=RANDOM_SEED
            )
        )
        inequality_rows.append(result)

inequality_table = pd.DataFrame(inequality_rows)
inequality_table.to_csv(OUTPUT_TABLE_DIR / "01_inequality_indices.csv", index=False)
display(inequality_table)

In [ ]:
decomposition = pd.DataFrame(
    [
        {
            "variable": "jumlah_lulusan",
            **theil_decomposition(
                kabupaten_summary,
                value_col="jumlah_lulusan",
                group_col="provinsi_pt",
            ),
        },
        {
            "variable": "jumlah_terdaftar",
            **theil_decomposition(
                kabupaten_summary,
                value_col="jumlah_terdaftar",
                group_col="provinsi_pt",
            ),
        },
        {
            "variable": "jumlah_pt",
            **theil_decomposition(
                kabupaten_summary,
                value_col="jumlah_pt",
                group_col="provinsi_pt",
            ),
        },
    ]
)
decomposition.to_csv(OUTPUT_TABLE_DIR / "01_theil_decomposition.csv", index=False)
display(decomposition)

In [ ]:
for frame, label, filename in [
    (province_summary, "Provinsi", "07_lorenz_province_graduates.png"),
    (kabupaten_summary, "Kabupaten/Kota", "08_lorenz_district_graduates.png"),
]:
    population, outcome = lorenz_coordinates(frame["jumlah_lulusan"])
    plt.figure(figsize=(7, 7))
    plt.plot(population, outcome, label=f"Kurva Lorenz — {label}")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Kesetaraan sempurna")
    plt.xlabel("Proporsi kumulatif unit wilayah")
    plt.ylabel("Proporsi kumulatif lulusan")
    plt.title(f"Kurva Lorenz jumlah lulusan: {label}")
    plt.legend()
    save_figure(filename)

## 7. Kesenjangan gender pada tingkat provinsi

Proporsi perempuan dihitung dari total lulusan perempuan dibagi total lulusan laki-laki dan perempuan. Kesenjangan intensitas membandingkan rasio agregat lulusan/terdaftar perempuan dengan laki-laki pada provinsi yang sama.

In [ ]:
province_gender = (
    gender_df.groupby(["provinsi_pt", "jk"], dropna=False, as_index=False)
    .agg(
        jumlah_lulusan=("jumlah_lulusan", "sum"),
        jumlah_terdaftar=("jumlah_terdaftar", "sum"),
    )
)
province_gender["graduate_output_intensity"] = safe_divide(
    province_gender["jumlah_lulusan"], province_gender["jumlah_terdaftar"]
)

graduate_pivot = province_gender.pivot(
    index="provinsi_pt", columns="jk", values="jumlah_lulusan"
).fillna(0)
enrolled_pivot = province_gender.pivot(
    index="provinsi_pt", columns="jk", values="jumlah_terdaftar"
)
intensity_pivot = province_gender.pivot(
    index="provinsi_pt", columns="jk", values="graduate_output_intensity"
)

province_gender_gap = pd.DataFrame(index=graduate_pivot.index)
province_gender_gap["lulusan_laki_laki"] = graduate_pivot.get("L", 0)
province_gender_gap["lulusan_perempuan"] = graduate_pivot.get("P", 0)
province_gender_gap["female_share_graduates"] = safe_divide(
    province_gender_gap["lulusan_perempuan"],
    province_gender_gap["lulusan_perempuan"] + province_gender_gap["lulusan_laki_laki"],
)
province_gender_gap["intensity_laki_laki"] = intensity_pivot.get("L")
province_gender_gap["intensity_perempuan"] = intensity_pivot.get("P")
province_gender_gap["female_minus_male_intensity"] = (
    province_gender_gap["intensity_perempuan"]
    - province_gender_gap["intensity_laki_laki"]
)
province_gender_gap["female_to_male_intensity_ratio"] = safe_divide(
    province_gender_gap["intensity_perempuan"],
    province_gender_gap["intensity_laki_laki"],
)
province_gender_gap = province_gender_gap.reset_index()
province_gender_gap.to_csv(
    OUTPUT_TABLE_DIR / "01_province_gender_gap.csv", index=False
)
display(province_gender_gap.sort_values("female_share_graduates", ascending=False).head(10))

In [ ]:
plot_data = province_gender_gap.sort_values("female_share_graduates")
plt.figure(figsize=(10, 9))
plt.barh(plot_data["provinsi_pt"], plot_data["female_share_graduates"] * 100)
plt.axvline(50, linestyle="--")
plt.xlabel("Proporsi perempuan di antara lulusan (%)")
plt.ylabel("Provinsi")
plt.title("Representasi perempuan di antara lulusan menurut provinsi")
save_figure("09_female_share_by_province.png")

## 8. Konsentrasi lulusan STEM

Analisis STEM menggunakan jumlah lulusan sebagai outcome utama. Rasio terhadap mahasiswa baru disajikan sebagai indikator keseimbangan arus masuk–keluar, bukan tingkat kelulusan cohort.

In [ ]:
province_stem_long = (
    stem_df.groupby(["provinsi_pt", "a_stem"], dropna=False, as_index=False)
    .agg(
        jumlah_lulusan=("jumlah_lulusan", "sum"),
        jumlah_mahasiswa_baru=("jumlah_mahasiswa_baru", "sum"),
    )
)
province_stem_long["graduate_to_entry_flow_ratio"] = safe_divide(
    province_stem_long["jumlah_lulusan"],
    province_stem_long["jumlah_mahasiswa_baru"],
)

stem_graduate_pivot = province_stem_long.pivot(
    index="provinsi_pt", columns="a_stem", values="jumlah_lulusan"
).fillna(0)
stem_entry_pivot = province_stem_long.pivot(
    index="provinsi_pt", columns="a_stem", values="jumlah_mahasiswa_baru"
)

province_stem_summary = pd.DataFrame(index=stem_graduate_pivot.index)
province_stem_summary["lulusan_stem"] = stem_graduate_pivot.get("STEM", 0)
province_stem_summary["lulusan_non_stem"] = stem_graduate_pivot.get("NON STEM", 0)
province_stem_summary["proporsi_lulusan_stem"] = safe_divide(
    province_stem_summary["lulusan_stem"],
    province_stem_summary["lulusan_stem"] + province_stem_summary["lulusan_non_stem"],
)
province_stem_summary["mahasiswa_baru_stem"] = stem_entry_pivot.get("STEM")
province_stem_summary["mahasiswa_baru_non_stem"] = stem_entry_pivot.get("NON STEM")
province_stem_summary["stem_graduate_to_entry_flow_ratio"] = safe_divide(
    province_stem_summary["lulusan_stem"],
    province_stem_summary["mahasiswa_baru_stem"],
)
province_stem_summary = province_stem_summary.reset_index()
province_stem_summary.to_csv(
    OUTPUT_TABLE_DIR / "01_province_stem_summary.csv", index=False
)
display(province_stem_summary.sort_values("proporsi_lulusan_stem", ascending=False).head(10))

In [ ]:
plot_data = province_stem_summary.dropna(subset=["proporsi_lulusan_stem"]).sort_values(
    "proporsi_lulusan_stem"
)
plt.figure(figsize=(10, 9))
plt.barh(plot_data["provinsi_pt"], plot_data["proporsi_lulusan_stem"] * 100)
plt.xlabel("Proporsi lulusan STEM (%)")
plt.ylabel("Provinsi")
plt.title("Proporsi lulusan STEM menurut provinsi")
save_figure("10_stem_share_by_province.png")

## 9. Pencilan wilayah secara robust

Pencilan dideteksi menggunakan *robust z-score* berbasis median dan MAD. Analisis dilakukan pada `log1p(jumlah_lulusan)` serta intensitas keluaran. Flag ini digunakan untuk pemeriksaan sensitivitas, bukan sebagai alasan otomatis untuk menghapus wilayah.

In [ ]:
outlier_table = kabupaten_summary.copy()
outlier_table["log1p_jumlah_lulusan"] = np.log1p(outlier_table["jumlah_lulusan"])
outlier_table["robust_z_log_graduates"] = robust_zscore(
    outlier_table["log1p_jumlah_lulusan"]
)
outlier_table["robust_z_output_intensity"] = robust_zscore(
    outlier_table["graduate_output_intensity"]
)
outlier_table["flag_outlier_size"] = outlier_table[
    "robust_z_log_graduates"
].abs().gt(3.5)
outlier_table["flag_outlier_intensity"] = outlier_table[
    "robust_z_output_intensity"
].abs().gt(3.5)

# Rasio dengan denominator sangat kecil dapat tampak ekstrem secara mekanis.
# Ambang 100 ditetapkan sebagai flag sensitivitas, bukan kriteria eksklusi otomatis.
outlier_table["flag_small_denominator_lt_100"] = outlier_table[
    "jumlah_terdaftar"
].lt(100)
outlier_table["flag_any_outlier"] = outlier_table[
    ["flag_outlier_size", "flag_outlier_intensity"]
].any(axis=1)
outlier_table["interpretation_status"] = np.select(
    [
        outlier_table["flag_outlier_intensity"]
        & outlier_table["flag_small_denominator_lt_100"],
        outlier_table["flag_any_outlier"],
    ],
    [
        "Periksa: outlier intensitas dengan denominator <100",
        "Periksa dalam sensitivity analysis",
    ],
    default="Tidak terflag",
)

outlier_table.sort_values(
    ["flag_any_outlier", "robust_z_log_graduates"], ascending=[False, False]
).to_csv(OUTPUT_TABLE_DIR / "01_district_outliers.csv", index=False)

display(
    outlier_table.loc[outlier_table["flag_any_outlier"]].sort_values(
        "robust_z_log_graduates", ascending=False
    )[
        [
            "provinsi_pt",
            "kabupaten_pt",
            "jumlah_lulusan",
            "jumlah_terdaftar",
            "lulusan_per_1000_terdaftar",
            "robust_z_log_graduates",
            "robust_z_output_intensity",
            "flag_outlier_size",
            "flag_outlier_intensity",
            "flag_small_denominator_lt_100",
            "interpretation_status",
        ]
    ]
)

In [ ]:
plot_data = outlier_table.dropna(
    subset=["jumlah_terdaftar", "jumlah_lulusan"]
)
plt.figure(figsize=(8, 6))
plt.scatter(
    np.log1p(plot_data["jumlah_terdaftar"]),
    np.log1p(plot_data["jumlah_lulusan"]),
    alpha=0.7,
)
plt.xlabel("log(1 + mahasiswa terdaftar)")
plt.ylabel("log(1 + jumlah lulusan)")
plt.title("Hubungan ukuran mahasiswa terdaftar dan jumlah lulusan")
save_figure("11_log_enrolled_vs_graduates.png")

## 10. Korelasi Spearman tingkat kabupaten/kota

Korelasi dihitung setelah variabel institusional dibuat unik per wilayah. Nilai *p* disesuaikan menggunakan prosedur Benjamini–Hochberg. Korelasi bersifat eksploratif dan tidak boleh ditafsirkan sebagai kausalitas.

In [ ]:
correlation_frame = kabupaten_summary.copy()
for source, destination in [
    ("jumlah_lulusan", "log1p_jumlah_lulusan"),
    ("jumlah_terdaftar", "log1p_jumlah_terdaftar"),
    ("jumlah_aktif_a", "log1p_jumlah_aktif"),
    ("jumlah_mbkm_m", "log1p_jumlah_mbkm"),
    ("jumlah_pt", "log1p_jumlah_pt"),
]:
    correlation_frame[destination] = np.log1p(correlation_frame[source])

correlation_targets = [
    "log1p_jumlah_lulusan",
    "graduate_output_intensity",
]
correlation_predictors = [
    "log1p_jumlah_terdaftar",
    "log1p_jumlah_aktif",
    "log1p_jumlah_mbkm",
    "log1p_jumlah_pt",
    "proporsi_mbkm",
    "proporsi_pt_akreditasi_tinggi",
    "proporsi_profesor",
    "proporsi_dosen_senior",
    "proporsi_dosen_tersertifikasi",
]

spearman_table = spearman_target_table(
    correlation_frame,
    targets=correlation_targets,
    predictors=correlation_predictors,
)
spearman_table.to_csv(
    OUTPUT_TABLE_DIR / "01_spearman_correlations.csv", index=False
)
display(spearman_table.sort_values(["target", "spearman_rho"], ascending=[True, False]))

In [ ]:
predictor_labels = {
    "log1p_jumlah_terdaftar": "Jumlah mahasiswa terdaftar (log1p)",
    "log1p_jumlah_aktif": "Jumlah mahasiswa aktif (log1p)",
    "log1p_jumlah_mbkm": "Jumlah peserta MBKM (log1p)",
    "log1p_jumlah_pt": "Jumlah perguruan tinggi (log1p)",
    "proporsi_mbkm": "Proporsi MBKM",
    "proporsi_pt_akreditasi_tinggi": "Proporsi PT akreditasi tinggi",
    "proporsi_profesor": "Proporsi profesor",
    "proporsi_dosen_senior": "Proporsi dosen senior",
    "proporsi_dosen_tersertifikasi": "Proporsi dosen tersertifikasi",
}
plot_data = spearman_table.loc[
    spearman_table["target"].eq("graduate_output_intensity")
].copy()
plot_data["predictor_label"] = plot_data["predictor"].map(predictor_labels)
plot_data = plot_data.sort_values("spearman_rho")
plt.figure(figsize=(10, 6))
plt.barh(plot_data["predictor_label"], plot_data["spearman_rho"])
plt.axvline(0, linewidth=1)
plt.xlabel("Koefisien Spearman")
plt.ylabel("Prediktor wilayah")
plt.title("Korelasi dengan intensitas keluaran lulusan")
save_figure("12_spearman_output_intensity.png")

## 11. Implikasi distribusi untuk pemodelan

Diagnostik berikut tidak menggantikan *model checking*, tetapi digunakan untuk mengunci keputusan awal model sebelum Notebook 02.

In [ ]:
model_diagnostics = pd.DataFrame(
    [
        {
            "analysis_level": "Strata gender–jenjang",
            "n": gender_df["jumlah_lulusan"].notna().sum(),
            "mean_target": gender_df["jumlah_lulusan"].mean(),
            "variance_target": gender_df["jumlah_lulusan"].var(ddof=1),
            "variance_to_mean": (
                gender_df["jumlah_lulusan"].var(ddof=1)
                / gender_df["jumlah_lulusan"].mean()
            ),
            "skewness": gender_df["jumlah_lulusan"].skew(),
            "zero_count": gender_df["jumlah_lulusan"].eq(0).sum(),
            "recommended_primary_model": "Negative Binomial dengan offset log(jumlah_terdaftar)",
            "validation_group": "provinsi_pt",
        },
        {
            "analysis_level": "Kabupaten/kota",
            "n": kabupaten_summary["jumlah_lulusan"].notna().sum(),
            "mean_target": kabupaten_summary["jumlah_lulusan"].mean(),
            "variance_target": kabupaten_summary["jumlah_lulusan"].var(ddof=1),
            "variance_to_mean": (
                kabupaten_summary["jumlah_lulusan"].var(ddof=1)
                / kabupaten_summary["jumlah_lulusan"].mean()
            ),
            "skewness": kabupaten_summary["jumlah_lulusan"].skew(),
            "zero_count": kabupaten_summary["jumlah_lulusan"].eq(0).sum(),
            "recommended_primary_model": "Negative Binomial; log1p target untuk ML pembanding",
            "validation_group": "provinsi_pt",
        },
    ]
)
model_diagnostics.to_csv(
    OUTPUT_TABLE_DIR / "01_model_distribution_diagnostics.csv", index=False
)
display(model_diagnostics)

## 12. Ringkasan otomatis dan checklist penyelesaian

In [ ]:
national_total = int(gender_df["jumlah_lulusan"].sum())
female_share = float(
    gender_summary.set_index("jk").loc["P", "share_lulusan"]
    if "P" in gender_summary["jk"].values
    else np.nan
)
stem_total = int(
    stem_df.loc[stem_df["a_stem"].eq("STEM"), "jumlah_lulusan"].sum()
)
stem_share = stem_total / stem_df["jumlah_lulusan"].sum()
province_gini = float(
    inequality_table.loc[
        (inequality_table["geography"].eq("Provinsi"))
        & (inequality_table["variable"].eq("jumlah_lulusan")),
        "gini",
    ].iloc[0]
)
district_gini = float(
    inequality_table.loc[
        (inequality_table["geography"].eq("Kabupaten/Kota"))
        & (inequality_table["variable"].eq("jumlah_lulusan")),
        "gini",
    ].iloc[0]
)
top_province = province_summary.sort_values("jumlah_lulusan", ascending=False).iloc[0]

summary_payload = {
    "data_source": DATA_SOURCE,
    "national_total_graduates_dataset_a": national_total,
    "national_total_graduates_dataset_b": int(stem_df["jumlah_lulusan"].sum()),
    "female_share_graduates": female_share,
    "stem_total_graduates": stem_total,
    "stem_share_graduates": stem_share,
    "province_gini_graduates": province_gini,
    "district_gini_graduates": district_gini,
    "largest_province_by_graduates": top_province["provinsi_pt"],
    "largest_province_graduates": int(top_province["jumlah_lulusan"]),
    "district_outlier_count": int(outlier_table["flag_any_outlier"].sum()),
    "random_seed": RANDOM_SEED,
}

with (OUTPUT_TABLE_DIR / "01_descriptive_summary.json").open("w", encoding="utf-8") as handle:
    json.dump(summary_payload, handle, ensure_ascii=False, indent=2)

summary_markdown = f"""# Ringkasan Deskriptif Otomatis\n\n- Total lulusan pada Dataset A: **{national_total:,}**.\n- Proporsi perempuan di antara lulusan: **{female_share:.2%}**.\n- Total lulusan STEM: **{stem_total:,}** atau **{stem_share:.2%}** dari seluruh lulusan.\n- Gini jumlah lulusan antarprovinsi: **{province_gini:.3f}**.\n- Gini jumlah lulusan antarkabupaten/kota: **{district_gini:.3f}**.\n- Provinsi dengan jumlah lulusan terbesar: **{top_province['provinsi_pt']}** ({int(top_province['jumlah_lulusan']):,}).\n- Kabupaten/kota yang teridentifikasi sebagai pencilan robust: **{int(outlier_table['flag_any_outlier'].sum())}**.\n\nAngka di atas bersifat agregat dan tidak boleh ditafsirkan sebagai probabilitas kelulusan individu.\n"""
(OUTPUT_TABLE_DIR / "01_descriptive_summary.md").write_text(
    summary_markdown, encoding="utf-8"
)

required_outputs = [
    "01_data_coverage.csv",
    "01_descriptive_statistics_strata.csv",
    "01_summary_by_gender.csv",
    "01_summary_by_level.csv",
    "01_kabupaten_summary.csv",
    "01_province_summary.csv",
    "01_inequality_indices.csv",
    "01_theil_decomposition.csv",
    "01_province_gender_gap.csv",
    "01_province_stem_summary.csv",
    "01_district_outliers.csv",
    "01_spearman_correlations.csv",
    "01_model_distribution_diagnostics.csv",
    "01_descriptive_summary.json",
    "01_descriptive_summary.md",
]
required_figures = [
    "01_distribution_graduates_raw.png",
    "02_distribution_graduates_log1p.png",
    "03_graduates_by_level.png",
    "04_female_share_by_level.png",
    "05_top15_provinces_graduates.png",
    "06_province_graduate_output_intensity.png",
    "07_lorenz_province_graduates.png",
    "08_lorenz_district_graduates.png",
    "09_female_share_by_province.png",
    "10_stem_share_by_province.png",
    "11_log_enrolled_vs_graduates.png",
    "12_spearman_output_intensity.png",
]

output_manifest = []
for filename in required_outputs:
    path = OUTPUT_TABLE_DIR / filename
    output_manifest.append(
        {"type": "table_or_summary", "filename": filename, "exists": path.exists(), "bytes": path.stat().st_size if path.exists() else 0}
    )
for filename in required_figures:
    path = OUTPUT_FIGURE_DIR / filename
    output_manifest.append(
        {"type": "figure", "filename": filename, "exists": path.exists(), "bytes": path.stat().st_size if path.exists() else 0}
    )
output_manifest = pd.DataFrame(output_manifest)
output_manifest.to_csv(OUTPUT_TABLE_DIR / "01_output_manifest.csv", index=False)

assert output_manifest["exists"].all(), "Ada output Notebook 01 yang tidak terbentuk."
assert np.isclose(
    gender_df["jumlah_lulusan"].sum(), 2_547_714
), "Total lulusan Dataset A berubah."
assert np.isclose(
    stem_df["jumlah_lulusan"].sum(), 2_547_717
), "Total lulusan Dataset B berubah."
assert inequality_table[["gini", "theil_t", "hhi"]].notna().all().all()

readiness_01 = pd.DataFrame(
    [
        {"check": "Dataset tervalidasi dapat dibaca", "status": "PASS", "evidence": DATA_SOURCE},
        {"check": "Agregasi wilayah tidak menggandakan indikator", "status": "PASS", "evidence": f"{len(kabupaten_summary):,} kabupaten/kota dan {len(province_summary):,} provinsi"},
        {"check": "Statistik deskriptif tersedia", "status": "PASS", "evidence": "01_descriptive_statistics_strata.csv"},
        {"check": "Indeks ketimpangan tersedia", "status": "PASS", "evidence": "Gini, Theil T, HHI, top-share, bootstrap CI"},
        {"check": "Kesenjangan gender terdokumentasi", "status": "PASS", "evidence": "01_province_gender_gap.csv"},
        {"check": "Konsentrasi STEM terdokumentasi", "status": "PASS", "evidence": "01_province_stem_summary.csv"},
        {"check": "Pencilan wilayah terdokumentasi", "status": "PASS", "evidence": "01_district_outliers.csv"},
        {"check": "Korelasi wilayah terdokumentasi", "status": "PASS", "evidence": "01_spearman_correlations.csv"},
        {"check": "Implikasi pemodelan ditetapkan", "status": "PASS", "evidence": "Negative Binomial + GroupKFold provinsi"},
        {"check": "Seluruh output terbentuk", "status": "PASS", "evidence": f"{len(output_manifest)} file output"},
    ]
)
readiness_01.to_csv(OUTPUT_TABLE_DIR / "01_readiness_checklist.csv", index=False)

display(pd.DataFrame([summary_payload]).T.rename(columns={0: "value"}))
display(readiness_01)

print("=" * 76)
print("NOTEBOOK 01 LOLOS.")
print("Analisis siap dilanjutkan ke 02_Negative_Binomial_Production_Function.ipynb")
print("=" * 76)